# Symbolic CSP network on CIFAR-10 (gradient-free)

The harder sibling of the MNIST notebook. CIFAR-10 (natural 32x32 colour
images) genuinely needs a **more sophisticated feature stack** than MNIST's
channel pooling, so the network is:

1. **Random convolutional feature bank** (Coates & Ng 2011 recipe,
   gradient-free): ZCA-whitened random image patches act as conv filters;
   rectified (both-sign) responses are quadrant-pooled into a feature
   vector. No backprop, no learned filters.
2. **Per-class symbolic readout**: `csp_sr` fits a sparse symbolic
   one-vs-rest scorer over those features; classify by `argmax`.

**Honest expectation.** This is a ceiling stress-test, not a SOTA claim.
Gradient-free random features + a symbolic readout land around **~55-70%**
on CIFAR-10; a *trained* CNN reaches ~95%. Accuracy is driven by the
(non-symbolic) feature bank; the symbolic head buys interpretability, not
perception. This is the empirical complement to the decomposition results:
**symbolic regression's value is discovery (physics/dynamics), not
vision** - here the symbolic head correctly degenerates to a sparse linear
readout, because image class-ness is a learned statistical pattern, not an
analytic law you can peel or factor.

## 1. Setup

In [ ]:
# Cell 1.1 - Setup: clone/refresh tessera to LATEST, install, verify.
import os, sys, subprocess
REPO_DIR = 'tessera'
REPO_URL = 'https://github.com/davechendatascience/tessera.git'

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode, r.stdout, r.stderr

if not os.path.exists(REPO_DIR):
    print('Cloning tessera...'); _run(['git', 'clone', REPO_URL])
else:
    print('Refreshing tessera to origin/main...')
    _run(['git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', 'main'])
    _run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'])
_, sha, _ = _run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s'])
print('tessera @', sha.strip())

print('Installing tessera[benchmarks,jax]...')
rc, out, err = _run([sys.executable, '-m', 'pip', 'install', '-e',
                     f'./{REPO_DIR}[benchmarks,jax]'])
if rc != 0:
    print('pip stderr tail:', err.splitlines()[-8:])
src_abs = os.path.abspath(os.path.join(REPO_DIR, 'src'))
if src_abs not in sys.path:
    sys.path.insert(0, src_abs)

import importlib
try:
    import tessera.experimental.csp_sr as _csp; importlib.reload(_csp)
    import tessera.experimental.csp_decompose as _cd; importlib.reload(_cd)
    assert hasattr(_csp, 'discover')
    print('OK: csp_sr + csp_decompose loaded.')
except Exception as e:
    print(f'LOAD FAILED ({e}). Restart runtime & re-run this cell.')

In [ ]:
# Cell 1.2 - Imports.
import time
import numpy as np
import matplotlib.pyplot as plt

from tessera.experimental.csp_sr import discover, CSPSRConfig, expr_to_str
from tessera.expression.tree import evaluate as eval_tree
print('csp_sr loaded')

try:
    import jax
    print(f'JAX devices: {jax.devices()}')
    _on_gpu = any('cuda' in str(d).lower() or 'gpu' in str(d).lower()
                  for d in jax.devices())
except Exception:
    _on_gpu = False
# The feature bank is numpy (fast already); use_jax only affects the head's
# Phi-build, which for a degree-1 (linear) head is trivial. Left off here.
USE_INTERPRETER = False
print('GPU detected.' if _on_gpu else 'CPU (fine for this notebook).')

## 2. Load CIFAR-10

In [ ]:
# Cell 2 - Load CIFAR-10 (keras, else torchvision), normalise, subset.
def load_cifar10():
    try:
        from tensorflow.keras.datasets import cifar10
        (Xtr, ytr), (Xte, yte) = cifar10.load_data()
        return Xtr, ytr.ravel(), Xte, yte.ravel()
    except Exception:
        import torchvision
        tr = torchvision.datasets.CIFAR10('./cifar', train=True, download=True)
        te = torchvision.datasets.CIFAR10('./cifar', train=False, download=True)
        return (np.asarray(tr.data), np.asarray(tr.targets),
                np.asarray(te.data), np.asarray(te.targets))

CLASSES = ['plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']
Xtr_full, ytr_full, Xte_full, yte_full = load_cifar10()

# Subset for speed. Set N_TRAIN=50000, N_TEST=10000 for the full run
# (slower; best on Colab). Random features improve with more train data.
N_TRAIN, N_TEST = 10000, 2000
rng = np.random.default_rng(0)
itr = rng.permutation(len(Xtr_full))[:N_TRAIN]
ite = rng.permutation(len(Xte_full))[:N_TEST]
Xtr = (Xtr_full[itr].astype(np.float32) / 255.0)
Xte = (Xte_full[ite].astype(np.float32) / 255.0)
ytr, yte = ytr_full[itr], yte_full[ite]

# Per-image contrast normalisation (helps random features a lot).
def contrast_norm(X):
    m = X.mean(axis=(1, 2, 3), keepdims=True)
    s = X.std(axis=(1, 2, 3), keepdims=True) + 1e-4
    return (X - m) / s
Xtr, Xte = contrast_norm(Xtr), contrast_norm(Xte)
print(f'CIFAR-10: train {Xtr.shape}, test {Xte.shape}, {len(CLASSES)} classes')

## 3. Random convolutional feature bank (gradient-free)

ZCA-whitened random image patches act as convolutional filters. Each image
is convolved (patch dot filter), responses are rectified on both signs
(`relu(z)` and `relu(-z)`), then mean-pooled over a `POOL x POOL` quadrant
grid. The result is a `POOL*POOL * 2K` feature vector per image. Everything
here is fixed/random - **no filter is learned**. This is the part that
actually drives CIFAR accuracy.

In [ ]:
# Cell 3 - Random-conv feature bank (Coates-Ng style), gradient-free.
PS, STRIDE, POOL, K = 6, 1, 2, 400     # patch, stride, pool grid, n filters

def _patches(imgs, ps, st):
    B, H, W, C = imgs.shape
    nh, nw = (H - ps) // st + 1, (W - ps) // st + 1
    out = np.empty((B, nh, nw, ps * ps * C), np.float32)
    for i in range(nh):
        for j in range(nw):
            out[:, i, j, :] = imgs[:, i*st:i*st+ps, j*st:j*st+ps, :].reshape(B, -1)
    return out, nh, nw

def fit_bank(imgs, ps, st, K, n_patch=100000, eps=0.1, seed=0):
    r = np.random.default_rng(seed)
    H, W = imgs.shape[1:3]
    ii = r.integers(0, len(imgs), n_patch)
    yy = r.integers(0, H - ps, n_patch); xx = r.integers(0, W - ps, n_patch)
    P = np.stack([imgs[ii[k], yy[k]:yy[k]+ps, xx[k]:xx[k]+ps, :].ravel()
                  for k in range(n_patch)]).astype(np.float32)
    mean = P.mean(0); Pc = P - mean
    U, S, _ = np.linalg.svd(Pc.T @ Pc / len(Pc))
    zca = (U @ np.diag(1.0 / np.sqrt(S + eps)) @ U.T).astype(np.float32)
    filt = Pc[r.integers(0, n_patch, K)] @ zca       # random whitened patches = filters
    filt /= np.linalg.norm(filt, axis=1, keepdims=True) + 1e-6
    return mean, zca, filt.astype(np.float32)

def bank_features(imgs, mean, zca, filt, ps, st, pool, bs=500):
    outs = []
    for s in range(0, len(imgs), bs):
        b = imgs[s:s+bs]; P, nh, nw = _patches(b, ps, st)
        Z = (P.reshape(-1, P.shape[-1]) - mean) @ zca
        A = Z @ filt.T
        A = np.concatenate([np.maximum(A, 0), np.maximum(-A, 0)], axis=1)  # both signs
        A = A.reshape(len(b), nh, nw, -1)
        gh, gw = nh // pool, nw // pool
        Ap = A[:, :gh*pool, :gw*pool].reshape(
            len(b), pool, gh, pool, gw, -1).mean(axis=(2, 4))
        outs.append(Ap.reshape(len(b), -1).astype(np.float32))
    return np.concatenate(outs, 0)

t0 = time.time()
mean, zca, filt = fit_bank(Xtr, PS, STRIDE, K)
Ftr = bank_features(Xtr, mean, zca, filt, PS, STRIDE, POOL)
Fte = bank_features(Xte, mean, zca, filt, PS, STRIDE, POOL)
# standardise features (helps the linear symbolic head's thresholding)
fmu, fsd = Ftr.mean(0), Ftr.std(0) + 1e-6
Ftr, Fte = (Ftr - fmu) / fsd, (Fte - fmu) / fsd
print(f'feature bank: {Ftr.shape[1]} features '
      f'(POOL {POOL}x{POOL} x 2*K={2*K}) in {time.time()-t0:.1f}s')

## 4. Joint multi-class head (one pass, regularised)

One-vs-rest fits 10 **independent** scorers against +/-1 - the wrong
objective (no shared multi-class competition) and, unregularised, it keeps
~all 3200 features and over-fits (train 0.93 / test 0.57). Instead we solve a
**single multi-output ridge** to one-hot labels:

`W = (X^T X + lam * I)^-1 X^T Y_onehot`

all 10 class columns at once (`X^T X` factored once), `lam` chosen on a
validation split. Gradient-free, one pass, regularised - this is the right
multi-class head, and it is far faster (one solve vs 10 SVD least-squares).

In [ ]:
# Cell 4 - Joint multi-class head: ONE regularised solve for all 10 classes.
def ridge_head(F, y, n_classes, lams=(1e0, 1e1, 1e2, 1e3, 1e4), val=0.2):
    N = len(F)
    A = np.c_[F, np.ones(N, np.float32)]
    nf = int(N * (1 - val))
    Af, yf, Av, yv = A[:nf], y[:nf], A[nf:], y[nf:]
    G = Af.T @ Af; AY = Af.T @ np.eye(n_classes, dtype=np.float32)[yf]
    I = np.eye(A.shape[1], dtype=np.float32)
    best = (-1.0, lams[0])
    for lam in lams:                                  # pick lam on validation
        W = np.linalg.solve(G + lam * I, AY)
        acc = float(((Av @ W).argmax(1) == yv).mean())
        if acc > best[0]:
            best = (acc, lam)
    lam = best[1]                                     # refit on ALL train
    Y = np.eye(n_classes, dtype=np.float32)[y]
    return np.linalg.solve(A.T @ A + lam * I, A.T @ Y), lam

t0 = time.time()
W, lam = ridge_head(Ftr, ytr, len(CLASSES))
def head_scores(F):
    return np.c_[F, np.ones(len(F), np.float32)] @ W
tr_acc = float((head_scores(Ftr).argmax(1) == ytr).mean())
preds = head_scores(Fte).argmax(1)
te_acc = float((preds == yte).mean())
print(f'Joint ridge head (lam={lam:g}): TRAIN={tr_acc:.4f}  TEST={te_acc:.4f}  '
      f'({time.time()-t0:.1f}s, ONE solve, gradient-free)')
# Each class score is a linear (symbolic) combination of features; show the
# few that dominate it (the interpretable readout).
for c in range(len(CLASSES)):
    top = np.abs(W[:-1, c]).argsort()[::-1][:5]
    print(f'  {CLASSES[c]:6s}: top-|w| features {list(top)}')

## 5. Network architecture

The full gradient-free pipeline as a DL-style layer graph: a fixed/random
convolutional feature bank feeds a single closed-form multi-class head.

In [ ]:
# Cell 5 - Architecture diagram (DL-style layer graph).
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(13, 3.0)); ax.axis('off')
layers = [
    ('Input\nCIFAR\n32x32x3', '#dfeefc'),
    (f'Random conv\nK={K} filt 6x6\nZCA-whitened', '#fde9d0'),
    ('Rectify +/-\n2K maps', '#fde9d0'),
    (f'{POOL}x{POOL} pool\n-> {Ftr.shape[1]} feats', '#fde9d0'),
    (f'Joint ridge head\n{Ftr.shape[1]} -> {len(CLASSES)}\n(one solve, lam={lam:g})', '#e7f6e1'),
    ('argmax\n10 classes', '#dfeefc'),
]
n = len(layers); w = 0.135; gap = (1 - 0.04 - n*w) / (n - 1); x0 = 0.02
for i, (label, color) in enumerate(layers):
    xc = x0 + i*(w + gap)
    ax.add_patch(FancyBboxPatch((xc, 0.30), w, 0.42,
                 boxstyle='round,pad=0.01', fc=color, ec='#555', lw=1.2))
    ax.text(xc + w/2, 0.51, label, ha='center', va='center', fontsize=8.5)
    if i < n - 1:
        ax.annotate('', xy=(x0 + (i+1)*(w+gap), 0.51), xytext=(xc + w, 0.51),
                    arrowprops=dict(arrowstyle='-|>', color='#555', lw=1.3))
ax.text(0.5, 0.93, f'Gradient-free symbolic CIFAR net  -  test acc {te_acc:.3f}',
        ha='center', fontsize=12, weight='bold', transform=ax.transAxes)
ax.text(0.30, 0.12, 'fixed / random  (no backprop)', ha='center',
        color='#c97a2b', fontsize=9, transform=ax.transAxes)
ax.text(0.80, 0.12, 'closed-form fit  (gradient-free)', ha='center',
        color='#3a8a3a', fontsize=9, transform=ax.transAxes)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); plt.tight_layout(); plt.show()

In [ ]:
# Cell 4.2 - Confusion matrix + per-class accuracy.
K_ = len(CLASSES)
cm = np.zeros((K_, K_), int)
for t, p in zip(yte, preds):
    cm[t, p] += 1
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(K_)); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticks(range(K_)); ax.set_yticklabels(CLASSES)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'CIFAR-10 symbolic net - test acc {te_acc:.3f}')
for i in range(K_):
    for j in range(K_):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=8)
plt.colorbar(im); plt.tight_layout(); plt.show()
per = cm.diagonal() / cm.sum(axis=1).clip(1)
for c in range(K_):
    print(f'  {CLASSES[c]:6s}: {per[c]:.3f}')

## What's next - honest scope

**What this is.** A gradient-free symbolic CIFAR-10 classifier: a random
convolutional feature bank (ZCA-whitened random-patch filters, rectified,
pooled) + a sparse symbolic linear readout per class. Every component is
fixed/random or closed-form; no backprop anywhere.

**The honest number.** Expect ~55-70% test accuracy (subset; a bit higher
with `N_TRAIN=50000`, more filters `K`, and a finer `POOL`). A *trained* CNN
reaches ~95%. The gap is the whole story: **on CIFAR, accuracy comes from
the features, and the best features are LEARNED (backprop)**, which this
deliberately does not do.

**Levers that raise the number** (still gradient-free): more filters `K`;
two stacked random-conv stages (a deeper random hierarchy); k-means filters
instead of random patches (Coates-Ng's stronger variant); finer pooling;
the full 50k training set. None change the conclusion - they push the
random-feature ceiling, not break it.

**Why the symbolic head stays linear.** As the decomposition study showed,
`csp_sr`/`discover_decompose` only finds deep structure where it exists
(physics: `sqrt(1-v^2/c^2)`-class, recovered exactly). Image class-ness has
no peelable outer op or variable-group separability, so the head correctly
degenerates to a capacity-controlled sparse linear readout - interpretable,
but not where the accuracy lives.

**The honest verdict, restated.** Symbolic regression is a *discovery* tool.
This notebook is the perception stress-test that marks the boundary: it works
(gradient-free, interpretable, ~60%), and it shows exactly why vision wants
learned features and physics wants symbolic ones.